# AutoQEC: 全自动 AI 量子纠错解码引擎 — 头脑风暴

> 本文档围绕 AutoQEC 项目进行系统性头脑风暴，覆盖核心架构、技术难点、差异化创新、风险评估与分阶段路线。

---
## 一、项目愿景

借鉴 Andrej Karpathy 等人倡导的自动化研究范式及开源 "AI Scientist" 思想，将大语言模型（LLM）的自动化研究能力（Auto-research）深度融合至量子纠错的底层物理中，构建一个全自动的 **AI 解码引擎（AutoQEC）**。

**核心输入：**
- 特定硬件的复杂噪声图谱
- 推理延迟约束

**核心输出：**
- 开箱即用的 AutoQEC 解码软件系统
- 自适应解码器算子库
- 位于不同 "准确率-延迟" Pareto 前沿上的最优预训练模型与推理算子

---
## 二、核心架构深层思考

### 2.1 "AI Scientist" 范式 → QEC 解码器设计的映射

| AI Scientist 阶段 | AutoQEC 对应操作 |
|---|---|
| 提出假设 | LLM 基于 Tanner 图拓扑 + 噪声图谱，生成解码器架构候选（NAS 搜索空间定义） |
| 设计实验 | 自动配置 Stim 参数：码距、噪声模型、采样量、评估指标 |
| 执行实验 | 闭环训练 Neural-BP 网络 + MWPM/OSD baseline 对比 |
| 分析结果 | 阈值计算、逻辑错误率拟合、Pareto 前沿更新 |
| 自我迭代 | LLM 读取实验报告，提出架构改进（层数、消息传递机制、注意力头等） |

**关键洞察：** QEC 解码器的搜索空间比通用 NAS 小得多——因为 Tanner 图的局部性约束天然限制了消息传递的拓扑。这使得 LLM-guided NAS 在此场景下的 **可行性远高于通用 CV/NLP NAS**。

### 2.2 混合解码范式设计

```
Syndrome Input
     |
     v
+------------------+
|  Neural-BP 前端   |  <-- 处理复杂串扰、空间关联噪声
|  (GNN on Tanner   |      可学习的消息传递调度
|   Graph)          |
+--------+---------+
         |  软判决输出 (marginal probabilities)
         v
+------------------+
|  置信度路由器     |  <-- 关键创新点！
|  (Confidence     |      判断 Neural-BP 是否"有信心"
|   Router)        |
+----+-------+-----+
     |       |
  高置信   低置信
     |       |
     v       v
  直接输出  MWPM/OSD
  (快速路径) (保底路径)
```

**置信度路由器** 是关键创新——它让系统在大多数简单情况下走快路径，少数困难情况下回退到经典算法。这直接服务于 Pareto 前沿目标。

---
## 三、技术难点与破解思路

### 3.1 难点 1：LLM 如何"理解"Tanner 图？

**问题：** LLM 不能直接"看"图结构，如何让它生成有意义的架构建议？

**思路：** 将 Tanner 图编码为 **文本描述 + 统计特征** 的混合表示：
- 度分布直方图（stabilizer 度、qubit 度）
- 码的参数 `[n, k, d]`、稳定子权重分布
- 噪声图谱的矩阵摘要（条件熵、互信息矩阵的低秩近似）
- 已知拓扑特征："这是 rotated surface code d=5" 或 "这是 [[144,12,12]] qLDPC 码"
- 借鉴 Graph-LLM 思路，用 GNN encoder 将图嵌入为 LLM 可消费的 token sequence

### 3.2 难点 2：NAS 搜索空间定义

QEC 解码器的架构搜索空间应围绕 Tanner 图的消息传递来定义：

```
搜索维度:
+-- 消息传递轮数 T in {3, 5, 8, 12, ...}
+-- 隐状态维度 d in {32, 64, 128, 256}
+-- 消息函数类型:
|   +-- MLP (标准)
|   +-- Attention-augmented (处理长程关联)
|   +-- Gated-Residual (处理过平滑)
|   +-- Hyperbolic GNN (处理层次化码结构)
+-- 归一化策略: BatchNorm / LayerNorm / GroupNorm
+-- 残差连接模式: standard / dense / weighted
+-- Readout 方式:
    +-- 直接 marginal argmax
    +-- Autoregressive bit-flip sequence
    +-- Energy-based (与 MWPM 目标对齐)
```

### 3.3 难点 3：合成数据 vs 真实芯片的 Domain Gap

**核心矛盾：** Stim 可以生成海量数据，但噪声模型是理想化的。真实芯片有：
- 空间非均匀噪声（某些 qubit 特别差）
- 时变噪声（热漂移、1/f 噪声）
- 测量错误的时间关联（leakage）
- 串扰模式比 depolarizing/pauli channel 复杂得多

**建议方案——三级噪声蒸馏流水线：**

```
Level 1: Stim 合成 (纯模拟)
  +-- 使用 Code Capacity / Phenomenological / Circuit-level 噪声模型
      参数化: {p, p_env, p_crosstalk_matrix, ...}

Level 2: 噪声模型校准 (Calibration -> Stim)
  +-- 从真实芯片的校准数据自动提取:
      - T1/T2 空间分布图
      - 单/双比特门错误率矩阵
      - 读出错误矩阵
      -> 自动生成 Stim 的自定义噪声通道

Level 3: Sim-to-Real 微调
  +-- 在 Level 1+2 预训练的解码器上
      用少量真实 syndrome-label 对做 LoRA-style 微调
      或者: 用域适应 (DANN) 对齐合成/真实 syndrome 的分布
```

---
## 四、差异化创新点（对比已有工作）

| 维度 | 已有工作 (e.g., DEFRAG, Breakdown, VEC-QEC) | AutoQEC 的差异 |
|---|---|---|
| **架构设计** | 人工设计 GNN/Transformer | LLM-guided NAS，自动搜索 |
| **噪声适应** | 单一噪声模型训练 | 合成预训练 + 真实校准自动微调 |
| **码适用范围** | 通常只针对 surface code | Tanner-graph-aware，天然扩展到 qLDPC |
| **部署** | 研究原型 | 开箱即用系统 + Pareto 最优算子库 |
| **迭代** | 人工调参 | 全闭环自我迭代 |

**最有价值的三大创新：**

1. **置信度路由器 + 混合解码** —— 让 AI 和经典算法各司其职
2. **基于 Tanner 图约束的 NAS** —— 比通用 NAS 高效，且保证搜索结果有物理意义
3. **Calibration-to-Stim 自动管线** —— 把真实芯片校准数据自动转化为训练数据

---
## 五、风险评估与缓解策略

### 高风险

| 风险 | 描述 | 缓解策略 |
|---|---|---|
| LLM 生成无效架构 | LLM 可能提出不符合物理约束的解码器结构 | 用 Tanner 图结构约束作为 hard filter；定义"架构合法性检查"函数 |
| MWPM/OSD 保底延迟高 | 大码距 qLDPC 上保底路径延迟可能不可接受 | 对保底路径做并行化/近似化（如 sparse OSD） |

### 中等风险

| 风险 | 描述 | 缓解策略 |
|---|---|---|
| 闭环迭代收敛性 | 自我迭代可能不收敛或陷入局部最优 | 设置上界（最多 N 轮）；每轮有阈值提升的 monotone guard |
| qLDPC Tanner 图复杂 | 非平面、长程连接导致消息传递困难 | 先在 surface code 验证再扩展；利用 LDPC 码理论研究收敛条件 |

### 低风险（需关注）

| 风险 | 缓解策略 |
|---|---|
| Stim 数据生成的 I/O 瓶颈 | 并行 Stim 进程 + 内存映射 |
| 模型部署量化/编译 | 提前考虑 ONNX/TensorRT 导出路径 |

---
## 六、分阶段项目路线

```
Phase 0: 基础设施
+-- Stim wrapper + 噪声模型参数化接口
+-- MWPM (PyMatching) + OSD baseline 集成
+-- 评估框架: 阈值曲线、延迟测量、Pareto 记录

Phase 1: 核心引擎 (Surface Code 验证)
+-- Neural-BP 基线实现 (在 d=3,5 surface code)
+-- NAS 搜索空间定义 + LLM prompt engineering
+-- 闭环: 实验 -> 结果分析 -> LLM 改进建议 -> 新实验
+-- 置信度路由器 v1

Phase 2: 扩展与自适应
+-- 扩展到 qLDPC 码 (如 [[144,12,12]])
+-- Calibration-to-Stim 自动管线
+-- Sim-to-Real 微调流程
+-- Pareto 前沿优化

Phase 3: 系统化交付
+-- 算子库打包 (预训练模型 + 推理算子)
+-- API 设计: 输入噪声图谱+延迟约束 -> 输出最优解码器
+-- 文档 + Benchmark 报告
```

---
## 七、开放问题

### 7.1 LLM 的选择
用 GPT-4/Claude 等 API，还是本地部署开源模型（如 DeepSeek、Qwen）？考虑到需要大量迭代调用，**成本和速率限制**是实际考量。

### 7.2 NAS 策略的粒度
在完整的 GNN 架构层面搜索，还是在更细粒度的"消息传递算子"层面搜索？后者搜索空间更小但可能更实用。

### 7.3 评估指标的权重
逻辑错误率 vs 推理延迟的 Pareto 前沿——"延迟"如何定义？是 GPU 推理延迟还是包括数据预处理的总延迟？目标场景是实时解码还是离线分析？

### 7.4 与硬件的耦合深度
做通用的解码器框架（适配任意超导/离子阱硬件），还是深度绑定某一特定硬件平台？

---

*Generated during brainstorming session for the AutoQEC project.*